# FEDrA — Exploratory Data Analysis (Step 3)

Loads `Dataset/manifest.csv` and covers:
1. Class distribution
2. Missing / corrupt files
3. URL length distribution per class
4. HTML file size distribution per class
5. Screenshot resolution consistency

All plots are saved to `notebooks/figures/`.

In [3]:
import os
import pathlib
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')          # non-interactive backend — safe for nbconvert
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from PIL import Image
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────────────────
ROOT       = pathlib.Path(os.getcwd()).parent   # FEDrA/
MANIFEST   = ROOT / 'Dataset' / 'manifest.csv'
FIGURES    = pathlib.Path('figures')            # relative to notebooks/
FIGURES.mkdir(exist_ok=True)

# ── Style ──────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.15)
PALETTE = {0: '#4CAF50', 1: '#F44336'}    # green = legit, red = phishing
LABEL_MAP = {0: 'Legit', 1: 'Phishing'}

df = pd.read_csv(MANIFEST)
df['label_name'] = df['label'].map(LABEL_MAP)
print(f'Manifest loaded: {MANIFEST}')
print(f'Shape          : {df.shape}')
df.head()

Manifest loaded: /Users/sunilkumar/Desktop/fedra/FEDrA/Dataset/manifest.csv
Shape          : (990, 6)


,sample_id,label,url,html_path,screenshot_path,label_name
0,0,0,github.com,C:\Users\User\OneDrive\Desktop\Mini Projects\M...,C:\Users\User\OneDrive\Desktop\Mini Projects\M...,Legit
1,1,0,apache.org,C:\Users\User\OneDrive\Desktop\Mini Projects\M...,C:\Users\User\OneDrive\Desktop\Mini Projects\M...,Legit
2,2,0,vimeo.com,C:\Users\User\OneDrive\Desktop\Mini Projects\M...,C:\Users\User\OneDrive\Desktop\Mini Projects\M...,Legit
3,3,0,en.wikipedia.org,C:\Users\User\OneDrive\Desktop\Mini Projects\M...,C:\Users\User\OneDrive\Desktop\Mini Projects\M...,Legit
4,4,0,wordpress.org,C:\Users\User\OneDrive\Desktop\Mini Projects\M...,C:\Users\User\OneDrive\Desktop\Mini Projects\M...,Legit


---
## 1 · Class Distribution

In [4]:
counts = df['label'].value_counts().sort_index()
labels = [LABEL_MAP[i] for i in counts.index]
colors = [PALETTE[i] for i in counts.index]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
fig.suptitle('Class Distribution', fontsize=15, fontweight='bold')

# Bar chart
bars = axes[0].bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.2, width=0.5)
axes[0].set_ylabel('Number of Samples')
axes[0].set_title('Sample Counts')
axes[0].yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
                 str(val), ha='center', va='bottom', fontweight='bold', fontsize=12)

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    counts.values, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for at in autotexts:
    at.set_fontsize(12)
axes[1].set_title('Class Split')

plt.tight_layout()
out = FIGURES / '01_class_distribution.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')
print(counts.to_string())
print(f'\nImbalance ratio (phishing:legit) = {counts[1]/counts[0]:.2f}:1')

Saved → figures/01_class_distribution.png
label
0    340
1    650

Imbalance ratio (phishing:legit) = 1.91:1


/var/folders/cf/14cpmw3x6w98rhwm4_nq_y6w0000gp/T/ipykernel_43958/1293180349.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 2 · Missing / Corrupt Files

In [5]:
def file_status(path_str):
    """Return ('ok'|'missing'|'empty') for a path string."""
    if not path_str or pd.isna(path_str):
        return 'missing'
    p = pathlib.Path(path_str)
    if not p.exists():
        return 'missing'
    if p.stat().st_size == 0:
        return 'empty'
    return 'ok'

def img_corrupt(path_str):
    """Return True if the image cannot be opened by Pillow."""
    if not path_str or pd.isna(path_str):
        return True
    try:
        with Image.open(path_str) as im:
            im.verify()
        return False
    except Exception:
        return True

print('Checking HTML files …')
df['html_status']  = [file_status(p) for p in tqdm(df['html_path'], leave=False)]
print('Checking screenshots …')
df['ss_status']    = [file_status(p) for p in tqdm(df['screenshot_path'], leave=False)]
print('Checking screenshot integrity …')
df['ss_corrupt']   = [img_corrupt(p) for p in tqdm(df['screenshot_path'], leave=False)]

print('\n── HTML file status ──')
print(df.groupby(['label_name','html_status']).size().unstack(fill_value=0).to_string())

print('\n── Screenshot status ──')
print(df.groupby(['label_name','ss_status']).size().unstack(fill_value=0).to_string())

print('\n── Corrupt screenshots ──')
print(df.groupby('label_name')['ss_corrupt'].sum().to_string())

# Visual summary
summary = pd.DataFrame({
    'Category': ['HTML Missing', 'HTML Empty', 'Screenshot Missing', 'Screenshot Empty', 'Screenshot Corrupt'],
    'Legit':    [
        (df[df.label==0]['html_status']=='missing').sum(),
        (df[df.label==0]['html_status']=='empty').sum(),
        (df[df.label==0]['ss_status']=='missing').sum(),
        (df[df.label==0]['ss_status']=='empty').sum(),
        df[df.label==0]['ss_corrupt'].sum()
    ],
    'Phishing': [
        (df[df.label==1]['html_status']=='missing').sum(),
        (df[df.label==1]['html_status']=='empty').sum(),
        (df[df.label==1]['ss_status']=='missing').sum(),
        (df[df.label==1]['ss_status']=='empty').sum(),
        df[df.label==1]['ss_corrupt'].sum()
    ]
})

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(summary))
width = 0.35
bars1 = ax.bar(x - width/2, summary['Legit'],    width, label='Legit',    color=PALETTE[0], edgecolor='white')
bars2 = ax.bar(x + width/2, summary['Phishing'], width, label='Phishing', color=PALETTE[1], edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(summary['Category'], rotation=15, ha='right')
ax.set_ylabel('Count')
ax.set_title('Missing / Corrupt Files per Class')
ax.legend()
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.2, str(int(h)),
                ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
out = FIGURES / '02_missing_corrupt_files.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

Checking HTML files …


Checking screenshots …


  0%|          | 0/990 [00:00<?, ?it/s]

OSError: [Errno 63] File name too long: 'C:\\Users\\User\\OneDrive\\Desktop\\Mini Projects\\Mini Project- 3rd Year\\FEDrA\\Dataset\\phised_dataset\\0442_cc68b94d-d9d0-4a03-bf37-d58a3335e1ce.p.reviewstudio.com_-_en_gp_profile_amzn1.account.ahu4k6zqt3fpraa57nl35ar56dna_ref_cm_cr_dp_d_gw_tr_ie_utf8\\screenshot.png'

---
## 3 · URL Length Distribution

In [6]:
df['url_len'] = df['url'].fillna('').apply(len)

print('── URL length statistics per class ──')
print(df.groupby('label_name')['url_len'].describe().round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('URL Length Distribution (chars)', fontsize=14, fontweight='bold')

# Overlapping KDE
for lbl, grp in df.groupby('label'):
    sns.kdeplot(grp['url_len'], ax=axes[0], label=LABEL_MAP[lbl],
                color=PALETTE[lbl], fill=True, alpha=0.35, linewidth=2)
axes[0].set_xlabel('URL Length (characters)')
axes[0].set_title('KDE — Density')
axes[0].legend()

# Box plot
box_data  = [df[df.label==l]['url_len'].values for l in [0, 1]]
bp = axes[1].boxplot(box_data, patch_artist=True, widths=0.4,
                     medianprops=dict(color='black', linewidth=2))
for patch, lbl in zip(bp['boxes'], [0, 1]):
    patch.set_facecolor(PALETTE[lbl])
    patch.set_alpha(0.75)
axes[1].set_xticklabels([LABEL_MAP[0], LABEL_MAP[1]])
axes[1].set_ylabel('URL Length (characters)')
axes[1].set_title('Box Plot')

plt.tight_layout()
out = FIGURES / '03_url_length_distribution.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

── URL length statistics per class ──
            count  mean   std   min   25%   50%   75%    max
label_name                                                  
Legit       340.0  15.4   6.1   4.0  10.0  15.0  20.0   34.0
Phishing    650.0  41.6  20.9  14.0  28.0  38.0  46.8  150.0
Saved → figures/03_url_length_distribution.png


/var/folders/cf/14cpmw3x6w98rhwm4_nq_y6w0000gp/T/ipykernel_43958/4281007910.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 4 · HTML File Size Distribution

In [7]:
def html_size_kb(path_str):
    if not path_str or pd.isna(path_str):
        return np.nan
    p = pathlib.Path(path_str)
    if not p.exists():
        return np.nan
    return p.stat().st_size / 1024   # kibibytes

print('Measuring HTML file sizes…')
df['html_kb'] = [html_size_kb(p) for p in tqdm(df['html_path'], leave=False)]

print('── HTML size (KB) statistics per class ──')
print(df.groupby('label_name')['html_kb'].describe().round(2).to_string())

valid = df.dropna(subset=['html_kb'])
cap   = valid['html_kb'].quantile(0.98)   # cap at 98th pct for readability
valid_cap = valid[valid['html_kb'] <= cap].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('HTML File Size Distribution (KB, capped at 98th pct)', fontsize=13, fontweight='bold')

for lbl, grp in valid_cap.groupby('label'):
    sns.kdeplot(grp['html_kb'], ax=axes[0], label=LABEL_MAP[lbl],
                color=PALETTE[lbl], fill=True, alpha=0.35, linewidth=2)
axes[0].set_xlabel('HTML Size (KB)')
axes[0].set_title('KDE — Density')
axes[0].legend()

box_data = [valid_cap[valid_cap.label==l]['html_kb'].values for l in [0, 1]]
bp = axes[1].boxplot(box_data, patch_artist=True, widths=0.4,
                     medianprops=dict(color='black', linewidth=2))
for patch, lbl in zip(bp['boxes'], [0, 1]):
    patch.set_facecolor(PALETTE[lbl])
    patch.set_alpha(0.75)
axes[1].set_xticklabels([LABEL_MAP[0], LABEL_MAP[1]])
axes[1].set_ylabel('HTML Size (KB)')
axes[1].set_title('Box Plot')

plt.tight_layout()
out = FIGURES / '04_html_size_distribution.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

Measuring HTML file sizes…


── HTML size (KB) statistics per class ──
            count  mean  std  min  25%  50%  75%  max
label_name                                           
Legit         0.0   NaN  NaN  NaN  NaN  NaN  NaN  NaN
Phishing      0.0   NaN  NaN  NaN  NaN  NaN  NaN  NaN
Saved → figures/04_html_size_distribution.png


/var/folders/cf/14cpmw3x6w98rhwm4_nq_y6w0000gp/T/ipykernel_43958/536817555.py:27: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  axes[0].legend()
/var/folders/cf/14cpmw3x6w98rhwm4_nq_y6w0000gp/T/ipykernel_43958/536817555.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 5 · Screenshot Resolution Consistency

In [8]:
def get_resolution(path_str):
    """Return (width, height) or (None, None) on failure."""
    if not path_str or pd.isna(path_str):
        return None, None
    try:
        with Image.open(path_str) as im:
            return im.size   # (width, height)
    except Exception:
        return None, None

print('Reading screenshot resolutions…')
resolutions = [get_resolution(p) for p in tqdm(df['screenshot_path'], leave=False)]
df['ss_w'] = [r[0] for r in resolutions]
df['ss_h'] = [r[1] for r in resolutions]

# ── Unique resolutions ──
res_counts = (df.dropna(subset=['ss_w','ss_h'])
                .assign(resolution=lambda d: d['ss_w'].astype(int).astype(str) + '×' + d['ss_h'].astype(int).astype(str))
                .groupby(['label_name','resolution']).size().reset_index(name='count')
                .sort_values('count', ascending=False))

print('\n── Top resolutions ──')
print(res_counts.head(20).to_string(index=False))

print('\n── Width statistics per class ──')
print(df.groupby('label_name')['ss_w'].describe().round(0).to_string())
print('\n── Height statistics per class ──')
print(df.groupby('label_name')['ss_h'].describe().round(0).to_string())

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Screenshot Resolution Consistency', fontsize=14, fontweight='bold')

# Width KDE
for lbl, grp in df.dropna(subset=['ss_w']).groupby('label'):
    sns.kdeplot(grp['ss_w'], ax=axes[0][0], label=LABEL_MAP[lbl],
                color=PALETTE[lbl], fill=True, alpha=0.35, linewidth=2)
axes[0][0].set_xlabel('Width (px)'); axes[0][0].set_title('Width Distribution'); axes[0][0].legend()

# Height KDE
for lbl, grp in df.dropna(subset=['ss_h']).groupby('label'):
    sns.kdeplot(grp['ss_h'], ax=axes[0][1], label=LABEL_MAP[lbl],
                color=PALETTE[lbl], fill=True, alpha=0.35, linewidth=2)
axes[0][1].set_xlabel('Height (px)'); axes[0][1].set_title('Height Distribution'); axes[0][1].legend()

# Scatter: width vs height
for lbl, grp in df.dropna(subset=['ss_w','ss_h']).groupby('label'):
    axes[1][0].scatter(grp['ss_w'], grp['ss_h'], alpha=0.4, s=15,
                       color=PALETTE[lbl], label=LABEL_MAP[lbl])
axes[1][0].set_xlabel('Width (px)'); axes[1][0].set_ylabel('Height (px)')
axes[1][0].set_title('Width vs Height Scatter'); axes[1][0].legend()

# Top-10 unique resolutions as horizontal bars
top10 = res_counts.head(10)
bar_colors = [PALETTE[0] if 'Legit' in r else PALETTE[1] for r in top10['label_name']]
axes[1][1].barh(top10['resolution'] + ' (' + top10['label_name'] + ')',
                top10['count'], color=bar_colors, edgecolor='white')
axes[1][1].set_xlabel('Count'); axes[1][1].set_title('Top-10 Unique Resolutions')
axes[1][1].invert_yaxis()

plt.tight_layout()
out = FIGURES / '05_screenshot_resolutions.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

Reading screenshot resolutions…



── Top resolutions ──
Empty DataFrame
Columns: [label_name, resolution, count]
Index: []

── Width statistics per class ──
           count unique  top freq
label_name                       
Legit          0      0  NaN  NaN
Phishing       0      0  NaN  NaN

── Height statistics per class ──
           count unique  top freq
label_name                       
Legit          0      0  NaN  NaN
Phishing       0      0  NaN  NaN


/var/folders/cf/14cpmw3x6w98rhwm4_nq_y6w0000gp/T/ipykernel_43958/933188487.py:37: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  axes[0][0].set_xlabel('Width (px)'); axes[0][0].set_title('Width Distribution'); axes[0][0].legend()
/var/folders/cf/14cpmw3x6w98rhwm4_nq_y6w0000gp/T/ipykernel_43958/933188487.py:43: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  axes[0][1].set_xlabel('Height (px)'); axes[0][1].set_title('Height Distribution'); axes[0][1].legend()
/var/folders/cf/14cpmw3x6w98rhwm4_nq_y6w0000gp/T/ipykernel_43958/933188487.py:50: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  axes[1][0].set_title('Width vs Hei

Saved → figures/05_screenshot_resolutions.png


/var/folders/cf/14cpmw3x6w98rhwm4_nq_y6w0000gp/T/ipykernel_43958/933188487.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Summary

In [9]:
total = len(df)
legit_n   = (df.label==0).sum()
phish_n   = (df.label==1).sum()
html_miss = (df['html_status'] != 'ok').sum()
ss_miss   = (df['ss_status']   != 'ok').sum()
ss_corr   = df['ss_corrupt'].sum()

print('╔══════════════════════════════════════════════════════╗')
print('║              FEDrA EDA Summary                       ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Total samples     : {total:<5}  (legit={legit_n}, phishing={phish_n})  ║')
print(f'║  Imbalance ratio   : {phish_n/legit_n:.2f}:1 (phishing:legit)        ║')
print(f'║  HTML issues       : {html_miss:<5} files missing or empty          ║')
print(f'║  Screenshot issues : {ss_miss:<5} missing/empty + {ss_corr} corrupt  ║')
print(f'║  URL avg length    : legit={df[df.label==0]["url_len"].mean():.0f} chars, phishing={df[df.label==1]["url_len"].mean():.0f} chars ║')
print(f'║  HTML avg size     : legit={df[df.label==0]["html_kb"].mean():.0f} KB, phishing={df[df.label==1]["html_kb"].mean():.0f} KB  ║')
print('╠══════════════════════════════════════════════════════╣')
print('║  Figures saved to notebooks/figures/                 ║')
print('║    01_class_distribution.png                         ║')
print('║    02_missing_corrupt_files.png                      ║')
print('║    03_url_length_distribution.png                    ║')
print('║    04_html_size_distribution.png                     ║')
print('║    05_screenshot_resolutions.png                     ║')
print('╚══════════════════════════════════════════════════════╝')
print('\n✅ EDA complete — ready for Step 4 (Feature Schema Decision)')

KeyError: 'ss_status'